# **Baseline Notebook**



---
## Setup Environment

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
# <Student to fill this section and then remove this comment>
group_name = "Neural Minds"
student_name = "Aryaman Malik"
student_id = "25980501"

In [ ]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [ ]:
!pip install -q scikit-learn
!pip install -q "vegafusion[embed]>=1.5.0"
!pip install -q "vl-convert-python>=1.6.0"

### 0.b Import Packages

In [ ]:
# <Student to fill this section and then remove this comment>
import pandas as pd
import altair as alt

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score,
                              confusion_matrix, precision_recall_curve, roc_curve)
# Enable vegafusion for Altair large dataset support
alt.data_transformers.enable("vegafusion")

---
## A. Assess Baseline Model

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load data
try:
  X_train = pd.read_csv(at.folder_path / 'X_train.csv')
  y_train = pd.read_csv(at.folder_path / 'y_train.csv')

  X_val = pd.read_csv(at.folder_path / 'X_val.csv')
  y_val = pd.read_csv(at.folder_path / 'y_val.csv')

  X_test = pd.read_csv(at.folder_path / 'X_test.csv')
  y_test = pd.read_csv(at.folder_path / 'y_test.csv')
except Exception as e:
  print(e)

### A.1 Generate Predictions with Baseline Model

In [ ]:
# A.1 Baseline anomaly-detection model: Isolation Forest with default parameters,
# contamination set to the observed anomaly rate of 5.58%.
#
# Isolation Forest isolates anomalies by randomly partitioning the feature space.
# Anomalies are isolated in fewer splits (shorter tree paths) than normal points,
# giving each line an anomaly score. The contamination parameter sets the threshold
# below which a line is flagged as anomalous.

# Make y a 1-D numpy vector regardless of how the CSV loaded it
y_train_1d = y_train.values.ravel() if hasattr(y_train, 'values') else np.asarray(y_train).ravel()
y_val_1d   = y_val.values.ravel()   if hasattr(y_val, 'values')   else np.asarray(y_val).ravel()
y_test_1d  = y_test.values.ravel()  if hasattr(y_test, 'values')  else np.asarray(y_test).ravel()

print(f"X_train: {X_train.shape}, y_train anomalies: {y_train_1d.sum()} ({y_train_1d.mean()*100:.2f}%)")
print(f"X_val:   {X_val.shape}, y_val anomalies:   {y_val_1d.sum()} ({y_val_1d.mean()*100:.2f}%)")
print(f"X_test:  {X_test.shape}, y_test anomalies:  {y_test_1d.sum()} ({y_test_1d.mean()*100:.2f}%)")

# Compute contamination from the observed training-set anomaly rate
contamination_rate = y_train_1d.mean()
print(f"\nContamination set to observed rate: {contamination_rate:.4f} ({contamination_rate*100:.2f}%)")

# Fit the Isolation Forest on the training set (UNSUPERVISED - it never sees y_train)
baseline_model = IsolationForest(
    contamination=contamination_rate,
    random_state=42,
    n_jobs=-1
)
baseline_model.fit(X_train)

# Helper: IF predict() returns -1 (anomaly) and 1 (normal); we map to 1/0
def to_anomaly_label(raw_pred):
    return (raw_pred == -1).astype(int)

pred_train = to_anomaly_label(baseline_model.predict(X_train))
pred_val   = to_anomaly_label(baseline_model.predict(X_val))
pred_test  = to_anomaly_label(baseline_model.predict(X_test))

# decision_function returns higher = more normal; flip sign so higher = more anomalous
score_train = -baseline_model.decision_function(X_train)
score_val   = -baseline_model.decision_function(X_val)
score_test  = -baseline_model.decision_function(X_test)

print(f"\nIsolation Forest flag rates:")
print(f"  train: {pred_train.mean()*100:.2f}%  (target: {contamination_rate*100:.2f}%)")
print(f"  val:   {pred_val.mean()*100:.2f}%")
print(f"  test:  {pred_test.mean()*100:.2f}%")

### A.2 Selection of Performance Metrics

> Provide some explanations on why you believe the performance metrics you chose is appropriate


In [ ]:
# A.2 Define the evaluation function we will use throughout the assignment.
# We report 5 complementary metrics suited to imbalanced anomaly detection.

def evaluate(name, y_true, y_pred, y_score):
    """Compute the full metric set for one split and pretty-print it."""
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred, zero_division=0)
    f1        = f1_score(y_true, y_pred, zero_division=0)
    roc_auc   = roc_auc_score(y_true, y_score) if len(np.unique(y_true)) > 1 else float('nan')
    pr_auc    = average_precision_score(y_true, y_score)
    cm        = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f"\n--- {name} ---")
    print(f"  Precision: {precision:.4f}   (of {tp+fp} lines flagged, {tp} were true anomalies)")
    print(f"  Recall:    {recall:.4f}   (caught {tp} of {tp+fn} true anomalies)")
    print(f"  F1:        {f1:.4f}   (harmonic mean of precision and recall)")
    print(f"  ROC-AUC:   {roc_auc:.4f}   (anomaly-vs-normal ranking quality)")
    print(f"  PR-AUC:    {pr_auc:.4f}   (precision-recall area; key metric for imbalanced data)")
    print(f"  Confusion matrix:")
    print(f"    TN={tn}  FP={fp}")
    print(f"    FN={fn}  TP={tp}")
    return {'precision':precision, 'recall':recall, 'f1':f1,
            'roc_auc':roc_auc, 'pr_auc':pr_auc,
            'tn':tn, 'fp':fp, 'fn':fn, 'tp':tp}

print("Metric definitions registered.")

In [ ]:
performance_metrics_explanations = """
We use 5 complementary metrics for this anomaly-detection problem (5.58% anomaly rate).

1. PRECISION = TP / (TP + FP)
   Of the lines we flagged as anomalies, what fraction were real anomalies? This is what an auditor cares about most - low precision wastes analyst time on false alarms.

2. RECALL = TP / (TP + FN)
   Of the real anomalies in the data, what fraction did the model catch? Missing real anomalies means undetected pricing errors and revenue leaks.

3. F1 = harmonic mean of precision and recall
   Single summary metric that punishes the model if EITHER precision OR recall is poor. F1 around 0.06 is what random guessing produces at this class rate; a useful model should reach 0.30+, a strong model 0.50+.

4. ROC-AUC
   Measures how well the model RANKS anomalies above normal lines, independent of the threshold. 0.5 is random; 1.0 is perfect. Useful but optimistic on imbalanced data because the score area is dominated by the majority class.

5. PR-AUC (average precision)
   Area under the precision-recall curve. Far more informative than ROC-AUC for rare-event detection because it focuses entirely on the positive class. A random model gets PR-AUC ~0.056 at our base rate; a useful model should reach 0.30+.

ACCURACY is deliberately EXCLUDED: predicting 'never anomaly' yields 94.4% accuracy on this data while being totally useless. Accuracy is misleading for imbalanced problems.

Primary metric for choosing the best model in the experiments notebook: F1 on the validation set. This balances precision and recall in one number and is the most aligned with the business goal of 'meaningful flags that an auditor can act on'. PR-AUC will be the secondary tiebreaker because it summarises the precision/recall trade-off across all possible thresholds. Test-set metrics are reserved for the final reported model only.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='performance_metrics_explanations', value=performance_metrics_explanations)

### A.3 Baseline Model Performance

> Provide some explanations on model performance


In [ ]:
# A.3 Run the metric set across all three splits
metrics_train = evaluate('TRAIN', y_train_1d, pred_train, score_train)
metrics_val   = evaluate('VAL',   y_val_1d,   pred_val,   score_val)
metrics_test  = evaluate('TEST',  y_test_1d,  pred_test,  score_test)

# Summary table
summary = pd.DataFrame({
    'Train': metrics_train, 'Val': metrics_val, 'Test': metrics_test
}).T[['precision','recall','f1','roc_auc','pr_auc','tp','fp','fn','tn']].round(4)
print("\n" + "="*60)
print("BASELINE SUMMARY: IsolationForest(contamination=0.0559, default params)")
print("="*60)
print(summary)

In [ ]:
# Visualise the precision-recall curve on the validation set
prec_vals, rec_vals, thresholds = precision_recall_curve(y_val_1d, score_val)
pr_df = pd.DataFrame({'recall': rec_vals, 'precision': prec_vals})
pr_chart = alt.Chart(pr_df).mark_line(color='#4C78A8', strokeWidth=2).encode(
    x=alt.X('recall:Q', title='Recall'),
    y=alt.Y('precision:Q', title='Precision', scale=alt.Scale(domain=[0,1])),
    tooltip=['recall','precision']
).properties(
    title=f'Validation Precision-Recall curve (PR-AUC = {metrics_val["pr_auc"]:.3f})',
    width=500, height=350
)

# Reference line: chance precision (= base rate)
chance_df = pd.DataFrame({'recall':[0,1], 'precision':[y_val_1d.mean(), y_val_1d.mean()]})
chance_line = alt.Chart(chance_df).mark_line(color='gray', strokeDash=[4,4]).encode(
    x='recall:Q', y='precision:Q'
)

(pr_chart + chance_line).display()

In [ ]:
baseline_performance_explanations = """
Isolation Forest baseline performance with default parameters and contamination set to the observed anomaly rate (5.59%):

Validation set (the number we will use to compare experiments against):
- Precision: 0.281    (vs 0.056 random) - 5.0x improvement over chance
- Recall:    0.281    (vs 0.056 random) - 5.0x improvement
- F1:        0.281    (vs 0.058 random) - 4.8x improvement
- ROC-AUC:   0.842    (vs 0.500 random) - strong ranking ability
- PR-AUC:    0.323    (vs 0.056 random) - 5.8x improvement

The model caught 132 of the 470 validation anomalies (28% recall) at 28% precision, meaning of the 470 lines it flagged, 132 were real anomalies and 338 were false alarms.

The validation precision-recall curve shows a clear two-regime behaviour: precision sits at 0.80-1.00 for the top-ranked 5-20% of flagged lines, drops sharply between 20% and 30% recall, then plateaus at around 0.15 - still well above the 0.056 chance baseline. This means the baseline is already highly useful at a strict threshold (an auditor reviewing only the top flags would see ~80% precision) but loses precision quickly as recall is pushed higher.

Test-set metrics (F1=0.328, ROC-AUC=0.848, PR-AUC=0.366) are slightly higher than val, which is reassuring - no overfitting to validation. Train F1 (0.326) is essentially the same as test, confirming the model is not overfitting (an off-the-shelf IF with n_estimators=100 has little capacity to memorise).

Why this is a good baseline:
1. It is the same family of algorithm we will tune in the experiments notebook, so improvements over baseline will be directly attributable to feature engineering, hyperparameter tuning, or alternative model choices - not to 'switching from a non-anomaly model to an anomaly model'.
2. ROC-AUC of 0.842 confirms the engineered features carry strong signal. The price/discount, margin, and subcategory-relative features have separated anomalies from normal lines in a way the unsupervised algorithm can exploit without seeing labels.
3. PR-AUC of 0.323 sets a meaningful bar. A lucky-threshold model could appear to do well at one operating point, but PR-AUC summarises performance across all thresholds and confirms real ranking ability.

Targets for the experiments notebook to beat:
- F1 above 0.40 on validation (1.4x baseline) - achievable through hyperparameter tuning.
- PR-AUC above 0.45 (1.4x baseline) - achievable by tuning IF or combining with other algorithms.
- ROC-AUC above 0.90 (modest improvement) - the baseline is already strong here, so big gains are unlikely.

Operational note: at the baseline's 28% precision, an auditor reviewing 100 flagged lines would find 28 real anomalies - a 5x productivity gain over uninformed sampling (which would yield ~6 hits per 100). Even before any tuning, the baseline IF is a deployable tool. The PR curve also shows that if the auditor instead reviews only the top 5-20% of flagged lines, precision rises to 80%+, meaning the baseline already supports two practical operating points: 'broad review' (28% precision, 28% recall) or 'high-confidence review' (~80% precision, ~15% recall). The experiments notebook will explore whether tuning and alternatives can lift both regimes simultaneously.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='baseline_performance_explanations', value=baseline_performance_explanations)